# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guided example for loading and exploring the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs. All references will use their `@id` fields for consistency.

_**Hint**: All references (record sets, fields, columns) use their Croissant `@id`s for clarity and reproducibility._

In [ ]:
# List all record sets available in this dataset and their fields by @id
record_set_ids = []
field_ids_by_record_set = {}

for rs in dataset.record_sets:
    record_set_ids.append(rs['@id'])
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        # If only one field, it's a dict
        fields = [fields]
    # field can be a list of dict, or list of @id strings
    f_ids = []
    for fld in fields:
        if isinstance(fld, dict) and '@id' in fld:
            f_ids.append(fld['@id'])
        elif isinstance(fld, str):
            f_ids.append(fld)
    field_ids_by_record_set[rs['@id']] = f_ids

print("Record sets (@id):")
for rid in record_set_ids:
    print(f"  - {rid}")

print("\nFields by record set (@id):")
for rid, fids in field_ids_by_record_set.items():
    print(f"  {rid}:")
    for fid in fids:
        print(f"    - {fid}")

## 3. Data Extraction

Load data from each record set using its `@id` into pandas DataFrames for analysis. All references use `@id` fields extracted above.

In [ ]:
# Extract data from each record set using its @id
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set {record_set_id}")
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

if dataframes:
    # Choose the largest (main) record set for further analysis
    selected_record_set_id = max(dataframes.keys(), key=lambda x: len(dataframes[x]))
    print(f"\nSelected main record set for analysis: {selected_record_set_id}")
    print("Available columns (@id):")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing data, and grouping data by attributes. All references use `@id` fields for fields and columns.

In [ ]:
# ----- Update the field @id here based on above data overview output -----
# For demo, we'll try 'age', 'Interval_between_Cancers', or another numeric column by its @id.
main_df = dataframes[selected_record_set_id]
# Find numeric columns by inspecting the first rows
numeric_candidates = main_df.select_dtypes(include=['number']).columns.tolist()
if not numeric_candidates:
    # Try to infer numeric columns by trying to convert some fields
    potential_numeric = []
    for col in main_df.columns:
        try:
            pd.to_numeric(main_df[col].dropna().iloc[:5])
            potential_numeric.append(col)
        except Exception:
            continue
    numeric_candidates = potential_numeric
print(f"Numeric field candidates (@id): {numeric_candidates}")
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Take the first numeric field by @id
    threshold = main_df[numeric_field].dropna().astype(float).mean() if main_df[numeric_field].dropna().shape[0]>0 else 0
    filtered_df = main_df[pd.to_numeric(main_df[numeric_field], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field], errors='coerce')
        - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try a grouping field (@id)
    # Pick a categorical field (try 'Sex', 'MSI_status', etc; fallback to first object column)
    object_candidates = main_df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for candidate in object_candidates:
        if candidate.lower() in ['sex', 'gender', 'msi_status', 'histology', 'anatomical_site', 'location']:
            group_field = candidate
            break
    if group_field is None and object_candidates:
        group_field = object_candidates[0]

    if group_field is not None:
        print(f"Grouping by field (@id): {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No categorical field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Here, we show a histogram of the selected numeric field and optionally a boxplot grouped by a categorical field, all using column `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals() and numeric_field in main_df:
    plt.figure(figsize=(8,5))
    sns.histplot(pd.to_numeric(main_df[numeric_field], errors='coerce').dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field} (@id)')
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals() and group_field in main_df:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field} (@id)')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library, referencing all entities strictly via their `@id` fields. This approach ensures reproducibility, clarity, and full traceability across all records, fields, and transformations—enabling confident downstream research and analysis for clinicopathological studies involving second primary colorectal cancer.

*Key steps completed:*  
- Loaded structured metadata and record sets via Croissant schema  
- Inspected all available record sets and fields by `@id`  
- Loaded records for each record set, and selected a main record set by size  
- Performed numeric field analysis (filtering, normalization, group-wise summary) using only `@id` identifiers  
- Visualized key distributions and relationships between clinical entities

Explore further: try alternate record sets, fields, or visualizations using `@id` references as needed. For dataset documentation and entity descriptions, refer to the [original Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).